# 01 — Load and profile the historical source

This notebook creates the Bronze working file. It reads the single retained source without modifying it, selects the fields used by the project, and shows you its first quality checks.

## Before you run

Open notebooks/analysis_settings.yaml. Keep row_limit: 5000 while learning. Change it to null only when you are ready to process the whole historical file.

In [ ]:
# Import the libraries and find the project root from any Jupyter working directory.
from pathlib import Path
import pandas as pd
import yaml


def find_project_root(start: Path) -> Path:
    """Return the folder that contains the notebooks/data folder."""
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "data").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project folder.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SETTINGS_PATH = PROJECT_ROOT / "notebooks" / "analysis_settings.yaml"
with SETTINGS_PATH.open(encoding="utf-8") as file:
    settings = yaml.safe_load(file)

SOURCE_PATH = PROJECT_ROOT / settings["source"]["path"]
BRONZE_PATH = PROJECT_ROOT / "notebooks" / "data" / "working" / "01_bronze_postings.csv"
print(f"Project root: {PROJECT_ROOT}")
print(f"Source file: {SOURCE_PATH}")
print(f"Row limit: {settings['run']['row_limit']}")

In [ ]:
# Stop early with a clear message if the one approved source file is missing.
if not SOURCE_PATH.exists():
    raise FileNotFoundError(
        f"The approved input is missing: {SOURCE_PATH}. "
        "Add the source file before running this notebook."
    )

# Read only the current source. nrows is None when row_limit is set to null in YAML.
row_limit = settings["run"]["row_limit"]
if row_limit is not None and (not isinstance(row_limit, int) or isinstance(row_limit, bool) or row_limit <= 0):
    raise ValueError("row_limit must be a positive integer or null.")
raw = pd.read_csv(SOURCE_PATH, nrows=row_limit, low_memory=False, dtype={"job_id": "string"})
if raw.empty:
    raise ValueError("The source contains no posting rows.")
if raw["job_id"].isna().any() or raw["job_id"].duplicated().any():
    raise ValueError("Source job IDs must be present and unique.")
print(f"Rows read: {len(raw):,}")
print(f"Columns found: {len(raw.columns)}")
raw.head(3)

In [ ]:
# Check that the source still has the fields this learning project expects.
required_columns = {
    "job_id", "company_name", "title", "description", "location",
    "formatted_work_type", "formatted_experience_level", "skills_desc",
    "min_salary", "max_salary", "med_salary", "remote_allowed", "currency", "pay_period", "listed_time",
    "job_posting_url", "application_url"
}
missing_columns = sorted(required_columns.difference(raw.columns))
if missing_columns:
    raise ValueError(f"The source schema changed. Missing columns: {missing_columns}")

# Show a simple null profile before any cleaning happens.
profile = (
    raw[list(required_columns)]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .rename("missing_percent")
    .to_frame()
)
profile

In [ ]:
# Select and rename only the columns that feed the rest of the project.
bronze = raw.rename(
    columns={
        "job_id": "source_job_id",
        "company_name": "company_raw",
        "title": "title_raw",
        "description": "description_raw",
        "location": "location_raw",
        "formatted_work_type": "employment_type_raw",
        "formatted_experience_level": "experience_label_raw",
        "skills_desc": "skills_raw",
        "min_salary": "salary_min_raw",
        "max_salary": "salary_max_raw",
        "med_salary": "salary_median_raw",
        "listed_time": "listed_time_raw",
        "job_posting_url": "job_url",
    }
)[[
    "source_job_id", "company_raw", "title_raw", "description_raw", "location_raw",
    "employment_type_raw", "remote_allowed", "experience_label_raw", "skills_raw", "salary_min_raw",
    "salary_max_raw", "salary_median_raw", "currency", "pay_period", "listed_time_raw", "job_url",
    "application_url"
]].copy()

# Add provenance columns once so downstream tables carry the source context.
bronze["source_name"] = settings["source"]["name"]
bronze["source_type"] = "historical_file"
bronze["collection_method"] = "retained_historical_snapshot"
bronze["source_snapshot_date"] = settings["source"]["snapshot_date"]
bronze["source_job_id"] = bronze["source_job_id"].astype("string")
bronze.head(3)

In [ ]:
# Save the Bronze file for notebook 02; this never overwrites the raw input.
BRONZE_PATH.parent.mkdir(parents=True, exist_ok=True)
bronze.to_csv(BRONZE_PATH, index=False)
print(f"Created {BRONZE_PATH.relative_to(PROJECT_ROOT)} with {len(bronze):,} rows.")

## Your checkpoint

Write three sentences here: what changed, one result you checked, and one limitation. Explain one code cell in your own words before opening the next notebook.